# 04 · Recomendador mejorado con metadatos (actores, director, palabras clave y géneros)

Continuación de `02_content_based.ipynb`. Aquí se implementa la sección **"Mejorando el recomendador"**
del tema *6. Sistemas recomendadores*:

1. Cargar y unir `credits.csv` (reparto y equipo) y `keywords.csv` con los metadatos.
2. Convertir las listas guardadas como texto a objetos de Python (`literal_eval`).
3. Extraer el **director** y los **3 actores/palabras clave/géneros** principales.
4. Limpiar (minúsculas y sin espacios) y construir la **"metadata soup"**.
5. Vectorizar con `CountVectorizer` y comparar con el recomendador de sinopsis (TF-IDF).
6. Extra sugerido por la lección: **filtro de popularidad** con la media ponderada de IMDB.

> **Nota de implementación:** la lección calcula la matriz de similitud completa
> (~45 000 × 45 000 números ≈ 16 GB de RAM), lo que no cabe en la mayoría de computadoras.
> Aquí se calcula **solo la fila de la película consultada**; el resultado es idéntico y usa muy poca memoria.

## 1. Carga de datos

In [1]:
import numpy as np
import pandas as pd
from ast import literal_eval

df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
credits = pd.read_csv('./data/credits.csv')
keywords = pd.read_csv('./data/keywords.csv')

print('movies_metadata:', df.shape)
print('credits        :', credits.shape)
print('keywords       :', keywords.shape)

movies_metadata: (45466, 24)
credits        : (45476, 3)
keywords       : (46419, 2)


Hay tres ids de `movies_metadata.csv` con formato de fecha en lugar de número (datos corruptos);
los eliminamos antes de convertir `id` a entero. Además, `credits` y `keywords` traen **ids repetidos**,
que al unirlos duplicarían películas; los quitamos para que una película no aparezca dos veces en las recomendaciones.

In [2]:
# Eliminar filas problemáticas (ids que no son numéricos)
df = df[pd.to_numeric(df['id'], errors='coerce').notna()].copy()

# Convertir todos los ids a números enteros
keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
df['id'] = df['id'].astype('int')

# Quitar duplicados antes de unir (evita películas repetidas)
df = df.drop_duplicates(subset='id')
credits = credits.drop_duplicates(subset='id')
keywords = keywords.drop_duplicates(subset='id')

# Hacer merges entre data frames
df = df.merge(credits, on='id')
df = df.merge(keywords, on='id')
df = df.reset_index(drop=True)
print('DataFrame final:', df.shape)
df[['title', 'cast', 'crew', 'keywords']].head(3)

DataFrame final: (45432, 27)


,title,cast,crew,keywords
0,Toy Story,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,Jumanji,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,Grumpier Old Men,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."


## 2. Recomendador base (solo sinopsis, TF-IDF)

Es el recomendador de la sección anterior: vectoriza la columna `overview` con TF-IDF y compara películas con la
**similitud coseno**. Como TF-IDF normaliza cada vector, el producto punto (`linear_kernel`) es igual a la similitud coseno y es más rápido.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

# Reemplazar NaN por string vacío y construir la matriz TF-IDF
df['overview'] = df['overview'].fillna('')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'])
print('Matriz TF-IDF:', tfidf_matrix.shape)

# Mapeo inverso: título -> índice
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

Matriz TF-IDF: (45432, 75827)


In [4]:
def get_recommendations(title, matrix, similarity=linear_kernel, n=10):
    """Devuelve las n películas más similares a `title`.

    Calcula la similitud solo entre la película pedida y todas las demás
    (una fila), en lugar de la matriz completa de N x N.

    Decisiones de diseño:
    - Una fila en vez de la matriz N x N: la matriz completa (~45 000 x 45 000 números) ocupa ~16 GB de RAM
      y no cabe en la mayoría de equipos; el resultado es idéntico y casi no usa memoria.
    - `similarity` es un parámetro porque cada matriz necesita una función distinta: TF-IDF ya está
      normalizada (basta `linear_kernel`, más rápido) y la de conteo no (hay que usar `cosine_similarity`).
    - Se excluye la propia película (`i != idx`) porque siempre es la más parecida a sí misma (similitud 1).
    """
    idx = indices[title]
    sim_scores = similarity(matrix[idx], matrix).ravel()
    # Ordenar de mayor a menor similitud e ignorar la propia película (posición 0)
    top = np.argsort(-sim_scores)
    top = [i for i in top if i != idx][:n]
    return df['title'].iloc[top]

In [5]:
get_recommendations('The Dark Knight Rises', tfidf_matrix)

12477                                      The Dark Knight
150                                         Batman Forever
1328                                        Batman Returns
15505                           Batman: Under the Red Hood
585                                                 Batman
21181    Batman Unmasked: The Psychology of the Dark Kn...
9228                    Batman Beyond: Return of the Joker
18027                                     Batman: Year One
19783              Batman: The Dark Knight Returns, Part 1
3094                          Batman: Mask of the Phantasm
Name: title, dtype: str

In [6]:
get_recommendations('The Godfather', tfidf_matrix)

1178               The Godfather: Part II
43999    The Godfather Trilogy: 1972-1990
1913              The Godfather: Part III
23110                          Blood Ties
11294                    Household Saints
34692                   Start Liquidation
10818                            Election
38003            A Mother Should Be Loved
17721                   Short Sharp Shock
26275                  Beck 28 - Familjen
Name: title, dtype: str

**Observación:** el recomendador de sinopsis encuentra películas con tramas parecidas, pero con calidad limitada.
*The Dark Knight Rises* devuelve casi solo películas de Batman, aunque a quien le gustó probablemente le gusten
otras películas de Christopher Nolan, algo que este modelo no puede capturar.

## 3. Extracción de datos

Las columnas `cast`, `crew`, `keywords` y `genres` están guardadas como **listas serializadas en texto**.
Hay que convertirlas a listas de diccionarios de Python con `literal_eval`.

In [7]:
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(literal_eval)

df[['title', 'cast', 'crew', 'keywords', 'genres']].head(2)

,title,cast,crew,keywords,genres
0,Toy Story,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."


In [8]:
def get_director(x):
    """Devuelve el nombre del director; NaN si no está listado.

    Decisión: se busca `job == 'Director'` dentro del equipo (`crew`) porque ahí vienen todos los puestos
    (productor, guionista...) y el director es un metadato clave: películas del mismo director suelen
    compartir estilo (p. ej. Nolan). Se devuelve NaN y no un texto inventado cuando falta el dato.
    """
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan


def get_list(x):
    """Devuelve los 3 primeros nombres de la lista, o todos si hay menos de 3.

    Decisión: solo 3 porque el reparto está ordenado por importancia, y los actores secundarios o los muchos
    keywords añaden ruido (dos películas no se parecen por compartir un extra) y agrandan el vocabulario.
    Si el dato no es una lista se devuelve [] para no romper el procesamiento posterior.
    """
    if isinstance(x, list):
        names = [i['name'] for i in x]
        # Si hay más de 3 elementos, regresar solo los primeros 3
        if len(names) > 3:
            names = names[:3]
        return names
    # Lista vacía si los datos no están bien formateados
    return []

In [9]:
# Extraer director de la columna crew
df['director'] = df['crew'].apply(get_director)

# Extraer top 3 de elenco, palabras clave y géneros
for feature in ['cast', 'keywords', 'genres']:
    df[feature] = df[feature].apply(get_list)

df[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,Jumanji,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",Joe Johnston,"[board game, disappearance, based on children'...","[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",Howard Deutch,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"


## 4. Pre-procesamiento y metadata soup

Se convierte todo a minúsculas y se **eliminan los espacios** dentro de cada nombre. Así el vectorizador
distingue `johnnydepp` de `johnnygalecki` (si no, contaría "Johnny" como la misma palabra en ambos) y no
confunde "bread jam" con "traffic jam".

In [10]:
def clean_data(x):
    """Pasa a minúsculas y elimina los espacios de nombres (o listas de nombres).

    Decisión: sin espacios, 'Johnny Depp' pasa a ser un solo término (`johnnydepp`); así el vectorizador no
    confunde a dos personas que comparten nombre de pila, y las minúsculas evitan contar 'Drama' y 'drama'
    como palabras distintas. Un director ausente se convierte en cadena vacía.
    """
    if isinstance(x, list):
        return [str.lower(i.replace(' ', '')) for i in x]
    # Si existe el director regresarlo limpio; si no, cadena vacía
    if isinstance(x, str):
        return str.lower(x.replace(' ', ''))
    return ''


for feature in ['cast', 'keywords', 'director', 'genres']:
    df[feature] = df[feature].apply(clean_data)

In [11]:
def create_soup(x):
    """Une keywords, actores, director y géneros en una sola cadena para el vectorizador.

    Decisión: se usan metadatos estructurados en vez de la sinopsis porque son más específicos y mejoran
    las recomendaciones (mismo director y mismo reparto). Una mejora posible, sugerida por la lección, es
    repetir al director varias veces en la 'sopa' para darle más peso.
    """
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])


df['soup'] = df.apply(create_soup, axis=1)
df[['title', 'soup']].head(3)

,title,soup
0,Toy Story,jealousy toy boy tomhanks timallen donrickles ...
1,Jumanji,boardgame disappearance basedonchildren'sbook ...
2,Grumpier Old Men,fishing bestfriend duringcreditsstinger walter...


## 5. CountVectorizer y similitud coseno

Se usa `CountVectorizer` y **no** TF-IDF: no queremos reducir el peso de un actor o director por haber trabajado
en muchas películas (el componente IDF haría justo eso, y aquí no tiene sentido intuitivo).
Como los vectores de conteo no están normalizados, se usa `cosine_similarity` en lugar de `linear_kernel`.

In [12]:
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])
print('Matriz de conteo:', count_matrix.shape)

Matriz de conteo: (45432, 73880)


In [13]:
# Reconstruir el mapeo título -> índice sobre el DataFrame final
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [14]:
get_recommendations('The Dark Knight Rises', count_matrix, similarity=cosine_similarity)

12477          The Dark Knight
10119            Batman Begins
9227                    Shiner
9783           Amongst Friends
7695                  Mitchell
23878                Quicksand
24819                 Deadfall
516          Romeo Is Bleeding
11351             The Prestige
42652    Lure: Teen Fight Club
Name: title, dtype: str

In [15]:
get_recommendations('The Godfather', count_matrix, similarity=cosine_similarity)

1913            The Godfather: Part III
1178             The Godfather: Part II
15432                   The Rain People
35462            Manuscripts Don't Burn
34248                              Rege
18742                         Last Exit
28450            In the Name of the Law
37997                 The Good Neighbor
7924     The Night of the Following Day
18063                 The Son of No One
Name: title, dtype: str

**Observación:** con los metadatos adicionales el recomendador captura mucho mejor la información. Para
*The Dark Knight Rises* aparecen *The Dark Knight*, *Batman Begins* y *The Prestige* (películas de Nolan), y para
*The Godfather* aparecen *Part II* y *Part III* junto con otras películas del mismo estilo, en lugar de títulos con la sinopsis parecida.

### Comparación lado a lado

In [16]:
def compare(title, n=10):
    return pd.DataFrame({
        'Solo sinopsis (TF-IDF)': get_recommendations(title, tfidf_matrix, n=n).values,
        'Metadatos (soup + CountVectorizer)': get_recommendations(title, count_matrix, cosine_similarity, n=n).values,
    })


compare('The Dark Knight Rises')

,Solo sinopsis (TF-IDF),Metadatos (soup + CountVectorizer)
0,The Dark Knight,The Dark Knight
1,Batman Forever,Batman Begins
2,Batman Returns,Shiner
3,Batman: Under the Red Hood,Amongst Friends
4,Batman,Mitchell
5,Batman Unmasked: The Psychology of the Dark Kn...,Quicksand
6,Batman Beyond: Return of the Joker,Deadfall
7,Batman: Year One,Romeo Is Bleeding
8,"Batman: The Dark Knight Returns, Part 1",The Prestige
9,Batman: Mask of the Phantasm,Lure: Teen Fight Club


In [17]:
compare('The Godfather')

,Solo sinopsis (TF-IDF),Metadatos (soup + CountVectorizer)
0,The Godfather: Part II,The Godfather: Part III
1,The Godfather Trilogy: 1972-1990,The Godfather: Part II
2,The Godfather: Part III,The Rain People
3,Blood Ties,Manuscripts Don't Burn
4,Household Saints,Rege
5,Start Liquidation,Last Exit
6,Election,In the Name of the Law
7,A Mother Should Be Loved,The Good Neighbor
8,Short Sharp Shock,The Night of the Following Day
9,Beck 28 - Familjen,The Son of No One


## 6. Mejora sugerida: filtro de popularidad

La lección propone: tomar las 30 películas más similares, calcular su **media ponderada de IMDB** y devolver
las 10 mejores. Así se evita recomendar películas desconocidas o mal valoradas solo porque se parecen.

$$WR = \frac{v}{v+m}\,R + \frac{m}{v+m}\,C$$

con *v* = votos de la película, *m* = mínimo de votos (percentil 90), *R* = calificación promedio y *C* = media global.

In [18]:
# Convertir las columnas de votos a número (algunas vienen como texto)
df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce')
df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce')

C = df['vote_average'].mean()
m = df['vote_count'].quantile(0.90)
print(f'C (calificación media global) = {C:.3f}   |   m (percentil 90 de votos) = {m:.0f}')


def weighted_rating(x, m=m, C=C):
    """Media ponderada de IMDB: WR = v/(v+m)*R + m/(m+v)*C.

    Decisiones:
    - Se usa WR y no `vote_average` crudo: una película con 9.0 de 10 votantes no debe superar a una con
      8.3 de 12 000; con pocos votos, WR la acerca a la media global C.
    - m = percentil 90 de votos (≈160), como en la lección: un umbral que se adapta al dataset en lugar
      de un número fijo arbitrario.
    """
    v, R = x['vote_count'], x['vote_average']
    return (v / (v + m) * R) + (m / (m + v) * C)


def improved_recommendations(title, matrix=count_matrix, similarity=cosine_similarity, n=10, candidates=30):
    """Toma las `candidates` más similares y las reordena por media ponderada.

    Decisiones:
    - Dos pasos (primero similitud, luego popularidad): así el resultado sigue siendo parecido a la película
      pedida, pero entre las parecidas se prefieren las bien valoradas y con muchos votos.
    - candidates=30 y n=10: valores sugeridos por la lección; con muy pocos candidatos el reordenamiento
      casi no cambiaría nada, y con demasiados se alejaría de la película original.
    """
    idx = indices[title]
    sims = similarity(matrix[idx], matrix).ravel()
    top = [i for i in np.argsort(-sims) if i != idx][:candidates]
    movies = df.iloc[top][['title', 'vote_count', 'vote_average']].copy()
    # Se descartan las películas sin dato de votos; las de pocos votos sí se califican,
    # pero WR las acerca a la media global y quedan al final
    movies = movies[movies['vote_count'].notna()]
    movies['score'] = movies.apply(weighted_rating, axis=1)
    return movies.sort_values('score', ascending=False).head(n)

C (calificación media global) = 5.618   |   m (percentil 90 de votos) = 160


In [19]:
improved_recommendations('The Dark Knight Rises')

,title,vote_count,vote_average,score
12477,The Dark Knight,12269.0,8.3,8.265480
11351,The Prestige,4510.0,8.0,7.918405
10119,Batman Begins,7511.0,7.5,7.460755
19459,Two Rabbits,39.0,7.1,5.908806
43960,Payback: Straight Up,25.0,6.8,5.778121
36517,Pitbull. New Order,20.0,6.7,5.738625
14508,Judex,18.0,6.6,5.717711
39867,Sara,12.0,7.0,5.714840
30324,Police Story 2013,92.0,5.8,5.684732
29882,Cold Summer of 1953,9.0,6.5,5.665399


In [20]:
improved_recommendations('The Godfather')

,title,vote_count,vote_average,score
1178,The Godfather: Part II,3418.0,8.3,8.180087
1165,Apocalypse Now,2112.0,8.0,7.832285
1913,The Godfather: Part III,1589.0,7.1,6.964467
3058,The Talented Mr. Ripley,790.0,7.1,6.850476
11621,The Consequences of Love,125.0,7.6,6.487553
5728,Thief,130.0,7.3,6.372250
1593,The Rainmaker,239.0,6.7,6.266297
3299,...And Justice for All,118.0,7.1,6.247311
37997,The Good Neighbor,116.0,6.2,5.862871
5260,The Gambler,21.0,7.0,5.778743


## Conclusiones
- El recomendador de **sinopsis** es simple, pero se limita a la trama.
- Con **actores, director, palabras clave y géneros** las recomendaciones reflejan mejor el gusto del usuario (mismo director, mismo estilo).
- El **filtro de popularidad** reordena los candidatos con la media ponderada para priorizar películas bien valoradas.
- Limitaciones: no considera las preferencias reales de cada usuario (eso requiere **filtrado colaborativo**), y los resultados dependen de la calidad de los metadatos.
- Posibles mejoras de la lección: incluir otros miembros del equipo (guionistas, productores) y dar más peso al director repitiéndolo en la "sopa".